In [1]:
import os
import threading
import h5py as h5
import numpy as np
import tensorflow as tf

# shuffle files
FILE_VALIDATE = 'shuffle_policy_validate.npz'
FILE_TRAIN = 'shuffle_policy_train.npz'
FILE_TEST = 'shuffle_policy_test.npz'

TRANSFORMATION_INDICES = {
    "noop": 0,
    "rot90": 1,
    "rot180": 2,
    "rot270": 3,
    "fliplr": 4,
    "flipud": 5,
    "diag1": 6,
    "diag2": 7
}

BOARD_TRANSFORMATIONS = {
    0: lambda feature: feature,
    1: lambda feature: np.rot90(feature, 1),
    2: lambda feature: np.rot90(feature, 2),
    3: lambda feature: np.rot90(feature, 3),
    4: lambda feature: np.fliplr(feature),
    5: lambda feature: np.flipud(feature),
    6: lambda feature: np.transpose(feature),
    7: lambda feature: np.fliplr(np.rot90(feature, 1))
}

def one_hot_action(action, size=19):
    """Convert an (x,y) action into a size x size array of zeros with a 1 at x,y
    """

    categorical = np.zeros((size, size))
    categorical[action] = 1
    return categorical

class threading_shuffled_hdf5_batch_generator:
    """A generator of batches of training data for use with the fit_generator function
       of Keras. Data is accessed in the order of the given indices for shuffling.

       it is threading safe but not multiprocessing therefore only use it with
       pickle_safe=False when using multiple workers
    """

    def shuffle_indices(self, seed=None, idx=0):
        # set generator_sample to idx
        self.metadata['generator_sample'] = idx

        # check if seed is provided or generate random
        if seed is None:
            # create random seed
            self.metadata['generator_seed'] = np.random.randint(1, 4294967295 + 1)

        # feed numpy.random with seed in order to continue with certain batch
        np.random.seed(self.metadata['generator_seed'])
        # shuffle indices according to seed
        if not self.validation:
            np.random.shuffle(self.indices)

    def __init__(self, state_dataset, action_dataset, indices, batch_size, metadata=None,
                 validation=False):
        self.action_dataset = action_dataset
        self.state_dataset = state_dataset
        # lock used for multithreaded workers
        self.data_lock = threading.Lock()
        self.indices_max = len(indices)
        self.validation = validation
        self.batch_size = batch_size
        self.indices = indices

        if metadata is not None:
            self.metadata = metadata
        else:
            # create metadata object
            self.metadata = {
                "generator_seed": None,
                "generator_sample": 0
            }

        # shuffle indices
        # when restarting generator_seed and generator_batch will
        # reset generator to the same point as before
        self.shuffle_indices(self.metadata['generator_seed'], self.metadata['generator_sample'])

    def __iter__(self):
        return self

    def next_indice(self):
        # use lock to prevent double hdf5 acces and incorrect generator_sample increment
        with self.data_lock:

            # get next training sample
            training_sample = self.indices[self.metadata['generator_sample'], :]
            # get state
            state = self.state_dataset[training_sample[0]]
            # get action
            # must be cast to a tuple so that it is interpreted as (x,y) not [(x,:), (y,:)]
            action = tuple(self.action_dataset[training_sample[0]])

            # increment generator_sample
            self.metadata['generator_sample'] += 1
            # shuffle indices when all have been used
            if self.metadata['generator_sample'] >= self.indices_max:
                self.shuffle_indices()

            # return state, action and transformation
            return state, action, training_sample[1]

    def __next__(self):
        game_size = self.state_dataset.shape[-1]
        state, action, transformation = self.next_indice()
        # get rotation symmetry belonging to state
        transform = BOARD_TRANSFORMATIONS[transformation]

        # get state from dataset and transform it.
        # loop comprehension is used so that the transformation acts on the
        # 3rd and 4th dimensions
        state_transform = np.array([transform(plane) for plane in state])
        state_transform = np.transpose(state_transform, (1, 2, 0))
        action_transform = transform(one_hot_action(action, game_size))

        return (state_transform, action_transform.flatten())


def load_indices_from_file(shuffle_file):
    # load indices from shuffle_file
    with open(shuffle_file, "rb") as f:
        indices = np.load(f)

    return indices

def remove_unused_symmetries(indices, symmetries):
    # remove all rows with a symmetry not in symmetries
    remove = []

    # find all rows with incorrect symmetries
    for row in range(len(indices)):
        if not indices[row][1] in symmetries:
            remove.append(row)

    # remove rows and return new array
    return np.delete(indices, remove, 0)



def load_train_val_test_indices(verbose, arg_symmetries, dataset_length, batch_size, directory):
    """Load indices from .npz files
       Remove unwanted symmerties
       Make Train set dividable by batch_size
       Return train/val/test set
    """
    # shuffle file locations for train/validation/test set
    shuffle_file_train = os.path.join(directory, FILE_TRAIN)
    shuffle_file_val = os.path.join(directory, FILE_VALIDATE)
    shuffle_file_test = os.path.join(directory, FILE_TEST)

    # load from .npz files
    train_indices = load_indices_from_file(shuffle_file_train)
    val_indices = load_indices_from_file(shuffle_file_val)
    test_indices = load_indices_from_file(shuffle_file_test)

    # used symmetries
    if arg_symmetries == "all":
        # add all symmetries
        symmetries = TRANSFORMATION_INDICES.values()
    elif arg_symmetries == "none":
        # only add standart orientation
        symmetries = [TRANSFORMATION_INDICES["noop"]]
    else:
        # add specified symmetries
        symmetries = [TRANSFORMATION_INDICES[name] for name in arg_symmetries.strip().split(",")]

    if verbose:
        print("Used symmetries: " + arg_symmetries)

    # remove symmetries not used during current run
    if len(symmetries) != len(TRANSFORMATION_INDICES):
        train_indices = remove_unused_symmetries(train_indices, symmetries)
        test_indices = remove_unused_symmetries(test_indices, symmetries)
        val_indices = remove_unused_symmetries(val_indices, symmetries)

    # Need to make sure training data is dividable by minibatch size or get
    # warning mentioning accuracy from keras
    if len(train_indices) % batch_size != 0:
        # remove first len(train_indices) % args.minibatch rows
        train_indices = np.delete(train_indices, [row for row in range(len(train_indices)
                                                  % batch_size)], 0)

    if verbose:
        print("dataset loaded")
        print("\t%d total positions" % dataset_length)
        print("\t%d total samples" % (dataset_length * len(symmetries)))
        print("\t%d total samples check" % (len(train_indices) +
              len(val_indices) + len(test_indices)))
        print("\t%d training samples" % len(train_indices))
        print("\t%d validation samples" % len(val_indices))
        print("\t%d test samples" % len(test_indices))

    return train_indices, val_indices, test_indices

2025-01-15 03:11:21.121511: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-15 03:11:21.133294: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-15 03:11:21.168807: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-15 03:11:21.226921: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-15 03:11:21.247013: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-15 03:11:21.273529: I tensorflow/core/platform/cpu_feature_gu

In [2]:
metadata = {
    "symmetries": "all",
    "batch_size": 16,
    "generator_seed": np.random.randint(1, 4294967295 + 1),
    "generator_sample": 0
}

out_directory = "../train/sl_policy"
verbose = True

In [3]:
# features of training data
dataset = h5.File("../train/sl_policy/feature_planes.h5")

In [4]:
# get train/validation/test indices
train_indices, val_indices, test_indices \
    = load_train_val_test_indices(verbose, metadata['symmetries'], len(dataset["states"]),
                                  metadata["batch_size"], out_directory)

Used symmetries: all
dataset loaded
	109979 total positions
	879832 total samples
	879831 total samples check
	835840 training samples
	43991 validation samples
	0 test samples


In [5]:
# create dataset generators
train_data_generator = threading_shuffled_hdf5_batch_generator(
    dataset["states"],
    dataset["actions"],
    train_indices,
    metadata["batch_size"],
    metadata)
val_data_generator = threading_shuffled_hdf5_batch_generator(
    dataset["states"],
    dataset["actions"],
    val_indices,
    metadata["batch_size"],
    validation=True)


def generate_train_data():
    while True:
        yield next(train_data_generator)

def generate_val_data():
    while True:
        yield next(val_data_generator)

In [6]:
for i, data in enumerate(generate_train_data()):
    print(data[0].shape, data[1].shape)
    if i > 10:
        break

(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)
(19, 19, 48) (361,)


In [7]:
output_signature = (tf.TensorSpec(shape=(19, 19, 48), dtype=tf.int32),
                    tf.TensorSpec(shape=(361), dtype=tf.int32))

tf_dataset = tf.data.Dataset.from_generator(generate_train_data, output_signature=output_signature).batch(metadata["batch_size"])

In [8]:
for i, element in enumerate(tf_dataset.as_numpy_iterator()):
  print(i, element[0].shape)
  if i > 10:
      break

0 (16, 19, 19, 48)
1 (16, 19, 19, 48)
2 (16, 19, 19, 48)
3 (16, 19, 19, 48)
4 (16, 19, 19, 48)
5 (16, 19, 19, 48)
6 (16, 19, 19, 48)
7 (16, 19, 19, 48)
8 (16, 19, 19, 48)
9 (16, 19, 19, 48)
10 (16, 19, 19, 48)
11 (16, 19, 19, 48)
